# Fine-tuning V8 — Data Augmentation

Igual ao `fine_tune.ipynb`, mas com augmentation aplicada em todas as imagens de treino.
Cada imagem recebe transformações aleatórias (flip, rotação, brilho, contraste),
garantindo que duplicatas do oversampling sejam versões distintas da imagem original.

### Configuração de ambiente

In [ ]:
from os import environ

environ['CUDA_VISIBLE_DEVICES'] = input('GPU ID: ')

### Imports

In [ ]:
from os.path import join
from json import load, dump
from datetime import timedelta
import random

from PIL import Image, ImageEnhance, ImageOps
from unsloth import FastVisionModel
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from tqdm.notebook import tqdm

import torch

from scripts.authentication import authenticate_huggingface
from scripts.data import SimpleLesionData, SimpleDatasetAnalysis
from scripts.messages import format_prompt, format_answer
from scripts.training import Training

import scripts.definitions as defs

### Autenticação

In [ ]:
authenticate_huggingface()

### Augmentation

Transformações aplicadas aleatoriamente em cada imagem de treino:
- Flip horizontal e vertical (50% de chance cada)
- Rotação em 0/90/180/270 graus
- Variação de brilho e contraste (±20%)

In [ ]:
def apply_augmentation(image: Image.Image) -> Image.Image:
    if random.random() > 0.5:
        image = ImageOps.mirror(image)

    if random.random() > 0.5:
        image = ImageOps.flip(image)

    angle = random.choice([0, 90, 180, 270])
    if angle:
        image = image.rotate(angle, expand=True)

    image = ImageEnhance.Brightness(image).enhance(random.uniform(0.8, 1.2))
    image = ImageEnhance.Contrast(image).enhance(random.uniform(0.8, 1.2))

    return image


def create_training_message_augmented(prompt_type: defs.PromptType,
                                      lesion_data: SimpleLesionData,
                                      dataset_analysis: SimpleDatasetAnalysis) -> dict:
    image_path = join(defs.DATA_PATH, 'stt_data', 'images', lesion_data.image)
    image = Image.open(image_path).convert('RGB')
    image = apply_augmentation(image)

    return {
        'messages': [
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'text',
                        'text': format_prompt(prompt_type, dataset_analysis),
                    },
                    {
                        'type': 'image',
                        'image': image,
                    }
                ],
            },
            {
                'role': 'assistant',
                'content': [{'type': 'text', 'text': format_answer(prompt_type, lesion_data)}],
            },
        ],
    }

### Configuração

In [ ]:
VERSION = 'V8'

training_hyperparameters = Training(
    base_model_name=defs.BASE_MODEL_NAME,
    trained_model_name=defs.MODEL_NAME,
    quantization=True,
    prompt_type=defs.PromptType.REPORT,
    version=VERSION,
    size=11,
    peft_hyperparameters={
        'finetune_vision_layers': True,
        'finetune_language_layers': True,
        'finetune_attention_modules': True,
        'finetune_mlp_modules': True,
        'r': 128,
        'lora_alpha': 128,
        'lora_dropout': 0.1,
        'bias': 'none',
        'random_state': defs.STATIC_RANDOM_STATE,
        'use_rslora': True,
        'loftq_config': None
    },
    sft_hyperparameters={
        'per_device_train_batch_size': 4,
        'gradient_accumulation_steps': 1,
        'learning_rate': 1e-4,
        'weight_decay': 0.01,
        'num_train_epochs': 2.0,
        'lr_scheduler_type': 'cosine',
        'warmup_ratio': 0.1,
        'optim': 'paged_adamw_32bit',
        'logging_steps': 1,
        'report_to': 'tensorboard',
        'output_dir': 'outputs',
        'seed': defs.STATIC_RANDOM_STATE,
        'bf16': is_bf16_supported(),
        'fp16': not is_bf16_supported(),
        'remove_unused_columns': False,
        'dataset_text_field': '',
        'dataset_kwargs': {'skip_prepare_dataset': True},
        'dataset_num_proc': 4,
        'max_seq_length': defs.MAX_TOKENS
    },
    used_memory=0.0,
    training_time=0.0
)

with open(join(defs.TRAINING_PATH, f'hyperparameters_{training_hyperparameters.version}.json'), 'w', encoding='utf-8') as file:
    dump(training_hyperparameters.model_dump(), file, indent=4, ensure_ascii=False)

### Carregamento do dataset

In [ ]:
with open(join(defs.DATA_PATH, 'stt_data', 'training_dataset.json'), 'r', encoding='utf-8') as file:
    training_dataset = [SimpleLesionData(**data) for data in load(file)]

with open(join(defs.DATA_PATH, 'training_dataset_analysis.json'), 'r', encoding='utf-8') as file:
    training_dataset_analysis = SimpleDatasetAnalysis(**load(file))

print(f'Dataset de treino: {len(training_dataset)} entradas')

### Preparação das mensagens com augmentation

In [ ]:
random.seed(defs.STATIC_RANDOM_STATE)

training_messages = []

for lesion_data in tqdm(training_dataset, desc='Criando mensagens com augmentation: '):
    training_messages.append(
        create_training_message_augmented(
            training_hyperparameters.prompt_type,
            lesion_data,
            training_dataset_analysis
        )
    )

### Inicialização do LLaMa 3.2

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    training_hyperparameters.base_model_name,
    load_in_4bit=training_hyperparameters.quantization,
    use_gradient_checkpointing='unsloth'
)

### Configuração de treinamento

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    **training_hyperparameters.peft_hyperparameters
)

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=training_messages,
    args=SFTConfig(**training_hyperparameters.sft_hyperparameters),
)

### Treinamento

In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f'Tempo de treinamento: {timedelta(seconds=trainer_stats.metrics["train_runtime"])}')
print(f'Memória máxima reservada: {used_memory} GB')

training_hyperparameters.used_memory = used_memory
training_hyperparameters.training_time = trainer_stats.metrics['train_runtime']

hyperparameters_name = f'hyperparameters_{training_hyperparameters.version}'

if training_hyperparameters.quantization:
    hyperparameters_name += '-4bit'

with open(join(defs.TRAINING_PATH, f'{hyperparameters_name}.json'), 'w', encoding='utf-8') as file:
    dump(training_hyperparameters.model_dump(), file, indent=4, ensure_ascii=False)

### Salvamento

In [ ]:
trained_model_name = f'{training_hyperparameters.trained_model_name}-{training_hyperparameters.version}-{training_hyperparameters.size}B'

if training_hyperparameters.quantization:
    trained_model_name += '-4bit'

if training_hyperparameters.prompt_type == defs.PromptType.SIMPLE_CLASSIFICATION:
    trained_model_name += '-SC'

save_path = join(defs.RESULTS_PATH, 'adapter_weights', trained_model_name)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

models_path = join(defs.TRAINING_PATH, 'models.json')

with open(models_path, 'r', encoding='utf-8') as file:
    content = file.read().strip()
    models = {name: defs.Model(**data) for name, data in __import__('json').loads(content).items()} if content else {}

new_model = defs.Model(
    local=True,
    quantized=training_hyperparameters.quantization,
    prompt_type=training_hyperparameters.prompt_type,
    version=training_hyperparameters.version,
    size=training_hyperparameters.size
)

models[trained_model_name] = new_model

for name, trained_model in models.items():
    models[name] = trained_model.model_dump()

with open(join(defs.TRAINING_PATH, 'models.json'), 'w', encoding='utf-8') as file:
    dump(models, file, indent=4, ensure_ascii=False)